# Mobility Matrix of a Rigid Assembly

Testing the `SoftPlankton` class with no degrees-of-freedom.

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
import plotly.express as px
from itertools import combinations

from softmobility import SoftPlankton

In [2]:
np.set_printoptions(linewidth=120, precision=3)

## Input with YAML format

In [3]:
yaml_data = """
param_names:
    - radexp
    - pos
    
defaults:
    radexp1: 0.75
    radexp2: 0.5
    pos1: 0.9
    pos2: 0.1
    pos3: 0.2
    pos6: 0.8

spheres:
  - # Sphere 0
    radius: 1.
    position: [0, 0, 0]
    
  - # Sphere 1
    radius: 0.1 * 10**radexp1
    position: 
      - 3 * (2*pos0 - 1)
      - 0
      - 0
    
  - # Sphere 2
    radius: 0.1 * 10**radexp2
    position: 
      - 3 * (2*pos1 - 1)
      - 3 * (2*pos2 - 1)
      - 0   
    
  - # Sphere 3
    radius: 0.1 * 10**radexp3
    position: 
      - 10 * (2*pos3 - 1)
      - 10 * (2*pos4 - 1)
      - 10 * (2*pos5 - 1)      
      
  - # Sphere 4
    radius: 0.1 * 10**radexp4
    position: 
      - 10 * (2*pos6 - 1)
      - 10 * (2*pos7 - 1)
      - 10 * (2*pos8 - 1)    
"""

In [4]:
# Creating SoftPlankton object
mysoftplankton = SoftPlankton(yaml_data)

  Found variables: pos0, pos1, pos2, pos3, pos4, pos5, pos6, pos7, pos8, radexp1, radexp2, radexp3, radexp4
    Sphere 0
      Radius: 1.00000000000000
      Position: ['0', '0', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 1
      Radius: 0.1*10**radexp1
      Position: ['6*pos0 - 3', '0', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 2
      Radius: 0.1*10**radexp2
      Position: ['6*pos1 - 3', '6*pos2 - 3', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 3
      Radius: 0.1*10**radexp3
      Position: ['20*pos3 - 10', '20*pos4 - 10', '20*pos5 - 10']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 4
      Radius: 0.1*10**radexp4
      Position: ['20*pos6 - 10', '20*pos7 - 10', '20*pos8 - 10']
      Orientation: ['0', '0', '0']
      Force:

## Calculating the mobility matrix

In [5]:
matrices = mysoftplankton.compute_mobility_problem()
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)
# print(M)
print(M_mean)

[[ 4.038e-02 -1.945e-03 -3.282e-04  3.257e-05  9.025e-04 -2.176e-03]
 [-1.945e-03  3.513e-02 -2.463e-03 -1.679e-03  6.562e-04  1.118e-03]
 [-3.282e-04 -2.463e-03  3.554e-02  2.135e-03 -1.168e-03 -6.522e-04]
 [ 3.257e-05 -1.679e-03  2.135e-03  1.434e-03 -1.919e-04 -1.378e-04]
 [ 9.025e-04  6.562e-04 -1.168e-03 -1.919e-04  2.203e-03  1.124e-03]
 [-2.176e-03  1.118e-03 -6.522e-04 -1.378e-04  1.124e-03  2.194e-03]]


In [6]:
dofs, params = mysoftplankton.dof_defaults, mysoftplankton.param_defaults
J = mysoftplankton.compute_composition_of_velocity(dofs, params)
R = mysoftplankton.compute_resistance_tensor(dofs, params)
Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
Mred = jnp.linalg.inv(J.T @ R @ J)
M_mean = Mred @ J.T @ Tf[:,:6]
print(M_mean)
print("Sphere's radii:", np.array([float(mysoftplankton.spheres[i].radius(dofs, params)) for i in range(mysoftplankton.Nspheres)]))

[[ 4.038e-02 -1.945e-03 -3.282e-04  3.257e-05  9.025e-04 -2.176e-03]
 [-1.945e-03  3.513e-02 -2.463e-03 -1.679e-03  6.562e-04  1.118e-03]
 [-3.282e-04 -2.463e-03  3.554e-02  2.135e-03 -1.168e-03 -6.522e-04]
 [ 3.257e-05 -1.679e-03  2.135e-03  1.434e-03 -1.919e-04 -1.378e-04]
 [ 9.025e-04  6.562e-04 -1.168e-03 -1.919e-04  2.203e-03  1.124e-03]
 [-2.176e-03  1.118e-03 -6.522e-04 -1.378e-04  1.124e-03  2.194e-03]]
Sphere's radii: [1.    0.562 0.316 0.1   0.1  ]


## Calculating angle of descent 

### Functions `calculation_angle`, `calculation_eigenvecs` and  `calculation_hydrodynamic_center`
It calculates the angle of descent based on shape parameters `params`and position of center of gravity `delta`.

In [7]:
def calculation_angle(params, delta):
    # calculation of mobility matrix
    # M, *_ = mysoftplankton.compute_mobility_problem(jnp.array([]), jnp.array([1., radii[0], radii[1], radii[2]]))
    # M_mean = jnp.mean(M.reshape(6, 6 * mysoftplankton.Nspheres).T.reshape(mysoftplankton.Nspheres, 6, 6), axis=0)
    dofs, params = jnp.array([]), jnp.array(params)
    J = mysoftplankton.compute_composition_of_velocity(dofs, params)
    R = mysoftplankton.compute_resistance_tensor(dofs, params)
    Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
    Mred = jnp.linalg.inv(J.T @ R @ J)
    M_mean = Mred @ J.T @ Tf[:,:6]
    # print("Initial mobility matrix:\n", M_mean)
    
    # calculation of submatrices
    mu_tt = M_mean[:3,:3]
    mu_tr = M_mean[:3,3:]
    mu_rt = M_mean[3:,:3]
    mu_rr = M_mean[3:,3:]
    assert np.allclose(mu_rt, mu_tr.T, atol=1e-5), "Mmean not symmetric"
    
    # Compute delta_cross (cross-product matrix)
    delta_cross = jnp.array([
        [0, -delta[2], delta[1]],
        [delta[2], 0, -delta[0]],
        [-delta[1], delta[0], 0]
    ])

    # Compute M1 and M2
    M2 = mu_rt + mu_rr @ delta_cross
    M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2
    print("Matrix M1\n", M1)
    print("Matrix M2\n", M2)
    assert np.allclose(M1, M1.T, atol=1e-5), "M1 not symmetric"
    
    # Compute complex eigenvalues and eigenvectors of M2
    eigvals1, _ = jnp.linalg.eigh(M1)  # General eigendecomposition
    eigvals2, eigvecs2 = jnp.linalg.eig(M2)  # General eigendecomposition
    assert np.all(eigvals1 >= 0), "M1 is not definite positive"

    # Boolean mask: which eigenvalues are real?
    real_mask = jnp.abs(jnp.imag(eigvals2)) < 1e-6

    # Replace non-real eigenvectors with zero columns
    real_eigvecs = eigvecs2[:,real_mask]  # Still (3,n), avoids dynamic indexing

    # Compute M1 * eigenvectors
    M1_eigvecs = M1 @ real_eigvecs  # (3,n)

    # Compute cos(angle) safely
    cos_angles = jnp.einsum("ij,ij->j", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(M1_eigvecs, axis=0)

    # Mask out invalid values
    cos_angles = jnp.real(cos_angles)
    print("Eigenvalues of M1:", eigvals1)
    print("Eigenvalues of M2:", eigvals2)
    print("Eigenvalues of M2 (real only):", eigvals2[real_mask])
    print("Cos angles:", cos_angles)
    print("Norm of eigenvectors", jnp.linalg.norm(real_eigvecs, axis=0))
    print("Eigenvectors", real_eigvecs)

    sin_angles = jnp.linalg.norm(jnp.cross(real_eigvecs,  M1_eigvecs, axis=0), axis=0) / jnp.linalg.norm(M1_eigvecs, axis=0)
    tan_angles = sin_angles / cos_angles
    # print("Sin angles:", sin_angles)
    # print("Tan angles:", tan_angles)

    print("Angles by inverting cos, sin, tan:")
    print((jnp.arccos(cos_angles)) * 180 / jnp.pi, "°")
    print((jnp.arcsin(sin_angles)) * 180 / jnp.pi, "°")
    print((jnp.arctan(tan_angles)) * 180 / jnp.pi, "°")
    
    def cos_angle(delta):
        # Compute delta_cross (cross-product matrix)
        delta_cross = jnp.array([
            [0, -delta[2], delta[1]],
            [delta[2], 0, -delta[0]],
            [-delta[1], delta[0], 0]
        ])

        # Compute M1 and M2
        M2 = mu_rt + mu_rr @ delta_cross
        M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2
        print("Matrix M1\n", M1)
        print("Matrix M2\n", M2)
        
        # Compute eigenvalues
        eigvals = jnp.linalg.eigvals(M2)

        # Boolean mask: which eigenvalues are real?
        real_mask = jnp.abs(jnp.imag(eigvals)) < 1e-6
        real_eigvals = jnp.where(real_mask, jnp.real(eigvals), 0.0)  # Replace complex ones with 0.0
        print("Eigenvalues of M2:", eigvals)
        print("Eigenvalues of M2 (real only):", real_eigvals)
            
        def solve_eigenvector(lmbda):
            """Finds an eigenvector using multiple cross products for robustness."""
            A = M2 - lmbda * jnp.eye(3)

            # Compute three different cross products
            v1 = jnp.cross(A[0], A[1])
            v2 = jnp.cross(A[0], A[2])
            v3 = jnp.cross(A[1], A[2])

            # Compute norms
            norm_v1 = jnp.linalg.norm(v1)
            norm_v2 = jnp.linalg.norm(v2)
            norm_v3 = jnp.linalg.norm(v3)

            # Choose the vector with the largest norm
            eigvec = jnp.where(norm_v1 >= norm_v2, v1, v2)
            eigvec = jnp.where(jnp.linalg.norm(eigvec) >= norm_v3, eigvec, v3)

            # Normalize and return
            return eigvec / (jnp.linalg.norm(eigvec) + 1e-16)  # Avoid NaNs
    
        real_eigvecs = jax.vmap(solve_eigenvector)(real_eigvals)  # (num_real, 3)
        print("Eigenvectors", real_eigvecs, jnp.linalg.norm(real_eigvecs, axis=1))

        # Compute M1 * eigenvectors
        M1_eigvecs = jnp.einsum("ij,kj->ik", M1, real_eigvecs)  # (3, num_real)

        # Compute cos(angle) safely
        cos_angles = jnp.einsum("ij,ji->i", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(M1_eigvecs, axis=0)

        # Compute min while ignoring zeroed-out values
        min_cos_angle = jnp.min(jnp.where(real_mask, cos_angles, 2.0))  # Set invalid ones to 2.0 (out of range)

        return min_cos_angle
    print("Jittable function, cos:", cos_angle(delta))

    return np.min(cos_angles)

In [8]:
def calculation_eigenvecs(params, delta):
    # calculation of mobility matrix
    dofs, params = jnp.array([]), jnp.array(params)
    J = mysoftplankton.compute_composition_of_velocity(dofs, params)
    R = mysoftplankton.compute_resistance_tensor(dofs, params)
    Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
    Mred = jnp.linalg.inv(J.T @ R @ J)
    M_mean = Mred @ J.T @ Tf[:,:6]
    
    # calculation of submatrices
    mu_tt = M_mean[:3,:3]
    mu_tr = M_mean[:3,3:]
    mu_rt = M_mean[3:,:3]
    mu_rr = M_mean[3:,3:]
    assert np.allclose(mu_rt, mu_tr.T, atol=1e-5), "Mmean not symmetric"
    
    # Compute delta_cross (cross-product matrix)
    delta_cross = jnp.array([
        [0, -delta[2], delta[1]],
        [delta[2], 0, -delta[0]],
        [-delta[1], delta[0], 0]
    ])

    # Compute M1 and M2
    M2 = mu_rt + mu_rr @ delta_cross
    M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2
    assert np.allclose(M1, M1.T, atol=1e-5), "M1 not symmetric"
    
    # Compute complex eigenvalues and eigenvectors of M2
    eigvals1, _ = jnp.linalg.eigh(M1)  # General eigendecomposition
    eigvals2, eigvecs2 = jnp.linalg.eig(M2)  # General eigendecomposition
    assert np.all(eigvals1 >= 0), "M1 is not definite positive"

    # Boolean mask: which eigenvalues are real?
    real_mask = jnp.abs(jnp.imag(eigvals2)) < 1e-8

    # Replace non-real eigenvectors with zero columns
    real_eigvecs = jnp.real(eigvecs2[:,real_mask])  # Still (3,n), avoids dynamic indexing

    # Compute M1 * eigenvectors
    M1_eigvecs = M1 @ real_eigvecs  # (3,n)
    velocity = M1_eigvecs / jnp.linalg.norm(M1_eigvecs, axis=0, keepdims=True) # (3, n) / (n,) = (3, n)

    # Compute cos(angle) safely
    cos_angles = jnp.einsum("ij,ij->j", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(M1_eigvecs, axis=0)

    # Mask out invalid values
    cos_angles = jnp.real(cos_angles)

    return real_eigvecs.T, velocity.T, cos_angles

In [9]:
def calculation_hydrodynamic_center(params):
    # calculation of mobility matrix
    dofs, params = jnp.array([]), jnp.array(params)
    M, *_ = mysoftplankton.compute_mobility_problem(dofs, params)
    M_mean = jnp.mean(M.reshape(6, 6 * mysoftplankton.Nspheres).T.reshape(mysoftplankton.Nspheres, 6, 6), axis=0)

    # submatrices to compute the mobility center
    b = M_mean[3:, :3]
    c = M_mean[3:, 3:]

    levicivita = np.array(
        [
            [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
            [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
            [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
        ]
    )

    # From Kim & Karila 5.3.2
    trcmcinv = np.linalg.inv(np.trace(c) * np.eye(3) - c)
    x_cm = 0.5 * np.einsum("ij,jkl,kl->i", trcmcinv, levicivita, b - b.transpose())

    return x_cm

### Example of usage

In [10]:
# Example of usage
params = jnp.linspace(0.5, 1, mysoftplankton.Nparam)
delta = calculation_hydrodynamic_center(params)
# delta = [1, 0.5, 0.5]
cos = calculation_angle(params, delta)
print("angle:", 180 * np.arccos(cos) / np.pi, "deg")
print(params, delta)

Matrix M1
 [[0.023 0.001 0.001]
 [0.001 0.024 0.001]
 [0.001 0.001 0.024]]
Matrix M2
 [[ 2.320e-05  3.534e-05 -1.162e-05]
 [ 3.534e-05  1.752e-05  5.949e-06]
 [-1.162e-05  5.949e-06 -2.369e-05]]
Eigenvalues of M1: [0.022 0.022 0.026]
Eigenvalues of M2: [ 5.608e-05+0.j -6.512e-06+0.j -3.253e-05+0.j]
Eigenvalues of M2 (real only): [ 5.608e-05+0.j -6.512e-06+0.j -3.253e-05+0.j]
Cos angles: [0.997 0.997 0.997]
Norm of eigenvectors [1. 1. 1.]
Eigenvectors [[ 0.74 +0.j  0.525+0.j  0.42 +0.j]
 [ 0.67 +0.j -0.63 +0.j -0.394+0.j]
 [-0.058+0.j -0.573+0.j  0.818+0.j]]
Angles by inverting cos, sin, tan:
[4.597 4.157 4.132] °
[4.597 4.157 4.132] °
[4.597 4.157 4.132] °
Matrix M1
 [[0.023 0.001 0.001]
 [0.001 0.024 0.001]
 [0.001 0.001 0.024]]
Matrix M2
 [[ 2.320e-05  3.534e-05 -1.162e-05]
 [ 3.534e-05  1.752e-05  5.949e-06]
 [-1.162e-05  5.949e-06 -2.369e-05]]
Eigenvalues of M2: [ 5.608e-05+0.j -6.512e-06+0.j -3.253e-05+0.j]
Eigenvalues of M2 (real only): [ 5.608e-05 -6.512e-06 -3.253e-05]
Eigenvec

## Optimizing the center of mass for given shape

### Function `optimize_delta` 
It optimizes the center of mass for a given shape defined by `params` using brute force exploration of all delta's. 

In [11]:
def optimize_delta(params, num_points):
    # calculation of mobility matrix
    dofs, params = jnp.array([]), jnp.array(params)
    J = mysoftplankton.compute_composition_of_velocity(dofs, params)
    R = mysoftplankton.compute_resistance_tensor(dofs, params)
    Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
    Mred = jnp.linalg.inv(J.T @ R @ J)
    M_mean = Mred @ J.T @ Tf[:,:6]
    
    # calculation of submatrices
    mu_tt = M_mean[:3,:3]
    mu_tr = M_mean[:3,3:]
    mu_rt = M_mean[3:,:3]
    mu_rr = M_mean[3:,3:]
    
    @jax.jit
    def cos_angle(delta):
        # Compute delta_cross (cross-product matrix)
        delta_cross = jnp.array([
            [0, -delta[2], delta[1]],
            [delta[2], 0, -delta[0]],
            [-delta[1], delta[0], 0]
        ])

        # Compute M1 and M2
        M2 = mu_rt + mu_rr @ delta_cross
        M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2

        # Compute eigenvalues
        eigvals = jnp.linalg.eigvals(M2)

        # Boolean mask: which eigenvalues are real?
        real_mask = jnp.abs(jnp.imag(eigvals)) < 1e-8
        real_eigvals = jnp.where(real_mask, jnp.real(eigvals), 0.0)  # Replace complex ones with 0.0

        @jax.jit
        def solve_eigenvector(lmbda):
            """Finds an eigenvector using multiple cross products for robustness."""
            A = M2 - lmbda * jnp.eye(3)

            # Compute three different cross products
            v1 = jnp.cross(A[0], A[1])
            v2 = jnp.cross(A[0], A[2])
            v3 = jnp.cross(A[1], A[2])

            # Compute norms
            norm_v1 = jnp.linalg.norm(v1)
            norm_v2 = jnp.linalg.norm(v2)
            norm_v3 = jnp.linalg.norm(v3)

            # Choose the vector with the largest norm
            eigvec = jnp.where(norm_v1 >= norm_v2, v1, v2)
            eigvec = jnp.where(jnp.linalg.norm(eigvec) >= norm_v3, eigvec, v3)

            # Normalize and return
            return eigvec / (jnp.linalg.norm(eigvec) + 1e-16)  # Avoid NaNs

        # Solve for all real eigenvalues (map function over them)
        real_eigvecs = jax.vmap(solve_eigenvector)(real_eigvals)  # (num_real, 3)

        # Compute M1 * eigenvectors
        M1_eigvecs = jnp.einsum("ij,kj->ik", M1, real_eigvecs)  # (3, num_real)

        # Compute cos(angle) safely
        cos_angles = jnp.einsum("ij,ji->i", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(M1_eigvecs, axis=0)

        # Compute min while ignoring zeroed-out values
        min_cos_angle = jnp.min(jnp.where(real_mask, cos_angles, 2.0))  # Set invalid ones to 2.0 (out of range)

        return min_cos_angle

    # Compute a, b, c based on the maximum absolute values of sphere positions
    positions = jnp.array([mysoftplankton.spheres[i].position(dofs, params) for i in range(mysoftplankton.Nspheres)])
    a = jnp.max(jnp.abs(positions[:, 0])) + 0.1
    b = jnp.max(jnp.abs(positions[:, 1])) + 0.1
    c = jnp.max(jnp.abs(positions[:, 2])) + 0.1

    # Create a grid of delta values inside the rectangular box [-a,a] × [-b,b] × [-c,c]
    delta_x_vals = jnp.linspace(-a, a, num_points)
    delta_y_vals = jnp.linspace(-b, b, num_points)
    delta_z_vals = jnp.linspace(-c, c, num_points)

    # Use indexing="ij" to keep dimensions consistent
    X, Y, Z = jnp.meshgrid(delta_x_vals, delta_y_vals, delta_z_vals, indexing="ij")

    # Flatten each dimension and stack into shape (num_points³, 3)
    deltas = jnp.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=-1)  # Shape: (num_points³, 3)

    # Compute cos_angle for all valid delta points
    cos_angles = jax.vmap(cos_angle)(deltas)

    # Find minimum cos_angle
    min_index = jnp.argmin(cos_angles)

    return deltas[min_index], cos_angles[min_index]

### Example of usage

In [12]:
params = jnp.linspace(0, 1, mysoftplankton.Nparam)
delta, cos = optimize_delta(params, 50)
print(cos, delta)

0.99605405 [-0.937  2.733  0.35 ]


In [13]:
calculation_angle(params,delta)

Matrix M1
 [[ 0.053 -0.007 -0.012]
 [-0.007  0.025  0.007]
 [-0.012  0.007  0.048]]
Matrix M2
 [[-0.005  0.002  0.008]
 [-0.005  0.001  0.003]
 [-0.009  0.003  0.004]]
Eigenvalues of M1: [0.023 0.039 0.065]
Eigenvalues of M2: [ 4.674e-05+0.007j  4.674e-05-0.007j -1.149e-04+0.j   ]
Eigenvalues of M2 (real only): [-0.+0.j]
Cos angles: [0.996]
Norm of eigenvectors [1.]
Eigenvectors [[-0.249+0.j]
 [-0.966+0.j]
 [ 0.075+0.j]]
Angles by inverting cos, sin, tan:
[5.092] °
[5.092] °
[5.092] °
Matrix M1
 [[ 0.053 -0.007 -0.012]
 [-0.007  0.025  0.007]
 [-0.012  0.007  0.048]]
Matrix M2
 [[-0.005  0.002  0.008]
 [-0.005  0.001  0.003]
 [-0.009  0.003  0.004]]
Eigenvalues of M2: [ 4.674e-05+0.007j  4.674e-05-0.007j -1.149e-04+0.j   ]
Eigenvalues of M2 (real only): [ 0.  0. -0.]
Eigenvectors [[-0.253 -0.965  0.069]
 [-0.253 -0.965  0.069]
 [-0.249 -0.966  0.075]] [1. 1. 1.]
Jittable function, cos: 0.99605405


Array(0.996, dtype=float32)

## 0ptimizing both shape and center of mass

### Contraints of no complete overlap of spheres and center-of-mass in the convex hull 

In [14]:
def create_constraint_functions(mysoftplankton):
    """Precompute and JIT-compile g and its Jacobian for a given plankton instance."""
    dofs = jnp.array([])

    def g(params):
        """Return only violated constraints g_ij."""
        positions = jnp.array([s.position(dofs, params) for s in mysoftplankton.spheres])
        radii = jnp.array([s.radius(dofs, params) for s in mysoftplankton.spheres])
        violations = []
        for i in range(len(mysoftplankton.spheres)):
            for j in range(i + 1, mysoftplankton.Nspheres):
                p_i, p_j = positions[i], positions[j]
                r_i, r_j = radii[i], radii[j]
                D_ij = jnp.linalg.norm(p_i - p_j)
                g_ij = (r_i + r_j) - D_ij
                violations.append(g_ij)

        return jnp.array(violations)

    # JIT-compile g and its Jacobian
    g_jit = jax.jit(g)
    jacobian_g_jit = jax.jit(jax.jacfwd(g))

    return g_jit, jacobian_g_jit

In [15]:
# Call this **once** before the optimization starts
g_jit, jacobian_g_jit = create_constraint_functions(mysoftplankton)

@jax.jit
def project_onto_nonoverlap(params):
    """JIT-optimized projection of params to satisfy non-overlapping constraints."""
    step_size = 1.01  # Fixed hyperparameters (JIT requires constants)
    eps = 1e-8
    max_iters = 100

    def cond_fn(state):
        params, i = state
        g_values = g_jit(params)
        return (jnp.max(g_values) > eps) & (i < max_iters)  # Stop early if constraints are satisfied

    def body_fn(state):
        params, i = state
        g_values = g_jit(params)
        J = jacobian_g_jit(params)

        weights = jnp.where(g_values > eps, 1.0, 0.0)  
        J_weighted = J * weights[:, None]  
        g_weighted = g_values * weights

        correction = step_size * (J_weighted.T @ g_weighted) / (1 + jnp.linalg.norm(J_weighted, axis=0))
        new_params = params - correction  

        return new_params, i + 1  

    params, _ = jax.lax.while_loop(cond_fn, body_fn, (params, 0))
    
    return params

In [16]:
def project_onto_convex_hull(delta, positions):
    """
    Projects delta onto an approximate convex hull using triangle faces.

    The function computes the intersection of the line from the assembly center 
    through delta with each triangle of the convex hull and finds the best projection.

    Parameters:
    - delta (array-like): The point to be projected.
    - positions (array-like): An array of 3D points defining the convex hull.

    Returns:
    - projected_delta (array-like): The projected point onto the convex hull.
    """

    # Convert positions to JAX array and compute the assembly center (mean position)
    positions = jnp.array(positions)
    assembly_center = jnp.mean(positions, axis=0)

    # Vector from the assembly center to delta
    to_delta = delta - assembly_center

    # Initialize min/max distances for projections
    min_distance = jnp.inf
    max_distance = -jnp.inf

    # Iterate over all triangle faces defined by point triplets
    for i, j, k in combinations(range(len(positions)), 3):
        p1, p2, p3 = positions[i], positions[j], positions[k]
        triangle_center = (p1 + p2 + p3) / 3

        # Compute the triangle's normal vector
        normal = jnp.cross(p2 - p1, p3 - p1)
        normal *= jnp.sign(jnp.dot(normal, triangle_center - assembly_center))  # Ensure outward direction
        normal /= jnp.linalg.norm(normal)

        # Compute the intersection of the line (assembly_center → delta) with the triangle's plane
        distance = jnp.dot(triangle_center - assembly_center, normal) / jnp.dot(to_delta, normal)
        intersection = assembly_center + distance * to_delta

        # Check if the intersection is inside the triangle using barycentric coordinates
        def is_inside_triangle(pt, p1, p2, p3):
            v0, v1, v2 = p3 - p1, p2 - p1, pt - p1
            d00, d01, d11 = jnp.dot(v0, v0), jnp.dot(v0, v1), jnp.dot(v1, v1)
            d20, d21 = jnp.dot(v2, v0), jnp.dot(v2, v1)
            denom = d00 * d11 - d01 * d01
            v = (d11 * d20 - d01 * d21) / denom
            w = (d00 * d21 - d01 * d20) / denom
            u = 1.0 - v - w
            return (u >= 0) & (v >= 0) & (w >= 0)

        # If the intersection lies within the triangle, update min/max distances
        if is_inside_triangle(intersection, p1, p2, p3):
            min_distance = jnp.minimum(min_distance, distance)
            max_distance = jnp.maximum(max_distance, distance)

    # Determine the final projection
    if (max_distance > 1) & (min_distance < 1):
        return delta  # Inside the convex hull, no need to project
    elif max_distance > 1:
        return assembly_center + min_distance * to_delta  # Project onto nearest boundary
    else:
        return assembly_center + max_distance * to_delta  # Project onto farthest boundary

### Function `optimize_delta_params`

In [17]:
def optimize_delta_params(
    init_params,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):
    """Optimizes both delta and radii using gradient descent with constraints.
    
    - Randomly samples `num_random_params` initial parameter sets.
    - Selects the best initial set by optimizing delta.
    - Performs gradient descent on both delta and parameters.
    """

    assert len(init_params==3 + mysoftplankton.Nparam), f"`init_params` has not the right lenght {3 + mysoftplankton.Nparam}"
    parameters = jnp.array(init_params)
    
    @jax.jit
    def cos_angle(parameters):
        """Computes cos(angle) for given delta and radii."""
        delta, params = parameters[:3], parameters[3:]
        dofs = jnp.array([])
        J = mysoftplankton.compute_composition_of_velocity(dofs, params)
        R = mysoftplankton.compute_resistance_tensor(dofs, params)
        Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
        Mred = jnp.linalg.inv(J.T @ R @ J)
        M_mean = Mred @ J.T @ Tf[:, :6]

        # Compute submatrices
        mu_tt = M_mean[:3, :3]
        mu_tr = M_mean[:3, 3:]
        mu_rt = M_mean[3:, :3]
        mu_rr = M_mean[3:, 3:]

        delta_cross = jnp.array([
            [0, -delta[2], delta[1]],
            [delta[2], 0, -delta[0]],
            [-delta[1], delta[0], 0],
        ])

        M2 = mu_rt + mu_rr @ delta_cross
        M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2

        # Compute eigenvalues
        eigvals = jnp.linalg.eigvals(M2)

        # Thresholds for imaginary and small real parts
        eps_imag = 1e-8  # Max allowed imaginary part
        eps_real = 1e-8  # Min allowed real part

        # Mask for nearly real eigenvalues (small imaginary part)
        real_mask = jnp.abs(jnp.imag(eigvals)) < eps_imag

        # Extract real parts
        real_eigvals = jnp.where(real_mask, jnp.real(eigvals), 0.0)

        # Apply a smooth penalty: replace small real eigenvalues smoothly
        penalized_eigvals = jnp.where(
            real_mask, 
            jnp.exp(- jnp.abs(real_eigvals) / eps_real), 
            1.
        )  # Discard non-real ones

        @jax.jit
        def solve_eigenvector(lmbda):
            """Finds an eigenvector using multiple cross products for robustness."""
            A = M2 - lmbda * jnp.eye(3)

            # Compute three different cross products
            v1 = jnp.cross(A[0], A[1])
            v2 = jnp.cross(A[0], A[2])
            v3 = jnp.cross(A[1], A[2])

            # Compute norms
            norm_v1 = jnp.linalg.norm(v1)
            norm_v2 = jnp.linalg.norm(v2)
            norm_v3 = jnp.linalg.norm(v3)

            # Choose the vector with the largest norm
            eigvec = jnp.where(norm_v1 >= norm_v2, v1, v2)
            eigvec = jnp.where(jnp.linalg.norm(eigvec) >= norm_v3, eigvec, v3)

            # Normalize and return
            return eigvec / (jnp.linalg.norm(eigvec) + 1e-16)  # Avoid NaNs

        # Compute eigenvectors for penalized eigenvalues
        real_eigvecs = jax.vmap(solve_eigenvector)(real_eigvals)
        M1_eigvecs = jnp.einsum("ij,kj->ik", M1, real_eigvecs)

        cos_angles = jnp.einsum("ij,ji->i", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(
            M1_eigvecs, axis=0
        )
        cos_angles = jnp.real(cos_angles) + penalized_eigvals

        # Find minimum cos(angle) after penalization
        min_cos_angle = jnp.min(jnp.where(real_mask, cos_angles, 2.0))  # Set invalid ones to 2.0 (out of range)

        return min_cos_angle

    def constraints_params(parameters):
        """Clips radii and enforces delta constraints within the convex hull of spheres."""
        delta, params = parameters[:3], parameters[3:]

        # Clip params within [0, 1]
        params = jnp.clip(params, 0, 1)

        # Apply projection to enforce non-overlapping constraint
        params = project_onto_nonoverlap(params)

        # Compute min/max bounds for delta (convex hull approximation)
        dofs = jnp.array([])
        all_positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
        all_radii = jnp.array([sphere.radius(dofs, params) for sphere in mysoftplankton.spheres])

        # Project delta onto convex hull of sphere's centers
        # delta = project_onto_convex_hull(delta, all_positions)

        # Compute axis-aligned bounding box (AABB)
        delta_min = 1. * jnp.min(all_positions - 1. * all_radii[:, None], axis=0)
        delta_max = 1. * jnp.max(all_positions + 1. * all_radii[:, None], axis=0)

        # Clip delta within the computed bounding box
        delta = jnp.clip(delta, delta_min, delta_max)

        return jnp.concatenate([delta, params])

    grad_cos_angle = jax.jit(jax.grad(cos_angle))
    velocity = jnp.zeros_like(parameters)  # Initialize velocity for momentum

    grad_threshold = 0.1  # Threshold for smoothing

    # Initialize best parameters tracking
    best_parameters = parameters
    best_cos_angle = cos_angle(parameters)

    for i in range(max_iters):
        grad = grad_cos_angle(parameters)
        grad_norm = jnp.linalg.norm(grad)

        # Smoothly scale gradient if too large
        scaling_factor = jnp.minimum(1.0, grad_threshold / (grad_norm + 1e-8))  # Smooth rescaling
        grad = grad * scaling_factor  

        # Compute new parameters
        velocity = momentum * velocity - learning_rate * grad
        parameters = parameters + velocity  # Update parameters

        # Enforce constraints
        parameters = constraints_params(parameters)
        
        # Compute current cos_angle
        current_cos_angle = cos_angle(parameters)

        # Track the best parameters based on minimum cos_angle
        if current_cos_angle < best_cos_angle:
            best_cos_angle = current_cos_angle
            best_parameters = parameters

        if i % 100 == 0:
            print(i, best_cos_angle, current_cos_angle, parameters, grad_norm)

    return best_parameters[:3], best_parameters[3:], best_cos_angle

### Trial run

In [52]:
delta, params, cos = optimize_delta_params(
    # num_random_params=1000,
    init_params=jnp.concatenate([jnp.array([0.1, 0.2, 0.3]), mysoftplankton.param_defaults]),
    learning_rate=0.1, 
    max_iters=10000
)
dofs = jnp.array([])
radii = np.array([mysoftplankton.spheres[i].radius(dofs, params) for i in range(mysoftplankton.Nspheres)])
print("\nOptimal radii:", radii)
print("params:", params)
print("delta, cos:", delta, cos)

0 0.9999795 0.9999795 [1.000e-01 2.000e-01 3.000e-01 0.000e+00 9.000e-01 9.999e-02 2.000e-01 4.815e-08 9.480e-07 8.000e-01 5.954e-07
 0.000e+00 7.500e-01 5.000e-01 1.225e-06 0.000e+00] 0.00025298167
100 0.99994355 0.99994355 [0.11  0.194 0.303 0.    0.883 0.102 0.185 0.003 0.001 0.788 0.003 0.    0.781 0.483 0.004 0.   ] 0.0016191226
200 0.9979516 0.9979516 [0.196 0.149 0.287 0.    0.882 0.251 0.07  0.057 0.025 0.705 0.054 0.019 1.    0.374 0.    0.   ] 0.0068935677
300 0.99698985 0.99698985 [0.253 0.11  0.258 0.063 0.899 0.478 0.    0.121 0.078 0.699 0.098 0.052 1.    0.466 0.    0.   ] 0.003799208
400 0.9963071 0.9963071 [0.289 0.086 0.239 0.079 0.88  0.533 0.    0.179 0.147 0.732 0.12  0.083 0.999 0.688 0.    0.   ] 0.003918614
500 0.99561447 0.99561447 [0.328 0.055 0.222 0.06  0.87  0.554 0.    0.26  0.246 0.78  0.151 0.13  0.984 0.893 0.    0.   ] 0.0033011374
600 0.99514526 0.99514526 [0.355 0.012 0.22  0.067 0.888 0.563 0.    0.369 0.371 0.86  0.2   0.196 0.971 0.934 0.    0.   

In [53]:
calculation_angle(params, delta)

Matrix M1
 [[ 2.828e-02  6.300e-05 -1.829e-04]
 [ 6.300e-05  2.334e-02  7.248e-04]
 [-1.829e-04  7.248e-04  2.162e-02]]
Matrix M2
 [[-2.901e-06 -4.868e-03 -1.778e-03]
 [ 4.925e-04  3.268e-06 -4.535e-04]
 [ 1.787e-04  4.537e-04 -3.699e-07]]
Eigenvalues of M1: [0.021 0.024 0.028]
Eigenvalues of M2: [ 5.655e-07+0.002j  5.655e-07-0.002j -1.133e-06+0.j   ]
Eigenvalues of M2 (real only): [-1.133e-06+0.j]
Cos angles: [0.99]
Norm of eigenvectors [1.]
Eigenvectors [[ 0.655+0.j]
 [-0.259+0.j]
 [ 0.709+0.j]]
Angles by inverting cos, sin, tan:
[8.024] °
[8.024] °
[8.024] °
Matrix M1
 [[ 2.828e-02  6.300e-05 -1.829e-04]
 [ 6.300e-05  2.334e-02  7.248e-04]
 [-1.829e-04  7.248e-04  2.162e-02]]
Matrix M2
 [[-2.901e-06 -4.868e-03 -1.778e-03]
 [ 4.925e-04  3.268e-06 -4.535e-04]
 [ 1.787e-04  4.537e-04 -3.699e-07]]
Eigenvalues of M2: [ 5.655e-07+0.002j  5.655e-07-0.002j -1.133e-06+0.j   ]
Eigenvalues of M2 (real only): [ 0.000e+00  0.000e+00 -1.133e-06]
Eigenvectors [[ 0.655 -0.26   0.71 ]
 [ 0.655 -0.26

Array(0.99, dtype=float32)

In [54]:
matrices = mysoftplankton.compute_mobility_problem(dofs, params)
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)
# print(M)
print(M_mean)

[[ 2.805e-02 -5.521e-06  2.093e-06 -9.757e-09  6.664e-07 -4.087e-06]
 [-5.521e-06  2.118e-02 -8.673e-09 -3.394e-06  2.326e-09  1.252e-05]
 [ 2.093e-06 -8.673e-09  2.118e-02  2.689e-05 -1.254e-05  5.104e-09]
 [-9.757e-09 -3.394e-06  2.689e-05  1.194e-02 -8.019e-06  2.482e-06]
 [ 6.664e-07  2.326e-09 -1.254e-05 -8.019e-06  1.207e-03 -1.863e-09]
 [-4.087e-06  1.252e-05  5.104e-09  2.482e-06 -1.863e-09  1.207e-03]]


In [55]:
dofs = jnp.array([])
radii = np.array([s.radius(dofs, params) for s in mysoftplankton.spheres])
positions = np.array([s.position(dofs, params) for s in mysoftplankton.spheres]) 
print("Optimal radii:\n", radii)
print("Optimal positions:\n", positions)
print("delta, cos:", delta, cos)

Optimal radii:
 [1.    0.97  0.968 0.761 0.759]
Optimal positions:
 [[ 0.000e+00  0.000e+00  0.000e+00]
 [-2.603e+00  0.000e+00  0.000e+00]
 [ 2.596e+00 -3.086e-03  0.000e+00]
 [-4.729e+00 -6.094e-04 -2.256e-03]
 [ 4.716e+00 -8.182e-03  6.828e-04]]
delta, cos: [ 0.366 -0.151  0.407] 0.9902104


## Visualization of the sphere assembly

### Useful plotting utility functions

In [18]:
def create_sphere(radius=1, center=(0, 0, 0), resolution=32):
    """
    Create a 3D checkered sphere with sharp transitions.
    """
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)

    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))

    return x, y, z

### Calculating centers of buoyancy, and sinking direction

In [56]:
# Compute geometric center (volume-weighted) / aka center of buoyancy
volumes = jnp.array([4/3 * jnp.pi * sphere.radius(dofs, params)**3 for sphere in mysoftplankton.spheres])
positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
geometric_center = jnp.sum(volumes[:, None] * positions, axis=0) / jnp.sum(volumes)

# Center of gravity
delta_point = jnp.sign(delta[2]) * jnp.array(delta)  

# Compute min/max ranges for each axis
all_positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
all_radii = jnp.array([sphere.radius(dofs, params) for sphere in mysoftplankton.spheres])

x_min, x_max = jnp.min(all_positions[:, 0] - all_radii), jnp.max(all_positions[:, 0] + all_radii)
y_min, y_max = jnp.min(all_positions[:, 1] - all_radii), jnp.max(all_positions[:, 1] + all_radii)
z_min, z_max = jnp.min(all_positions[:, 2] - all_radii), jnp.max(all_positions[:, 2] + all_radii)

# Use a consistent range to ensure 1:1:1 aspect ratio
max_range = max(x_max - x_min, y_max - y_min, z_max - z_min) / 2.0
mid_x, mid_y, mid_z = (x_max + x_min) / 2, (y_max + y_min) / 2, (z_max + z_min) / 2

# Choose a colormap
colormap = "Viridis"  # Try "Turbo", "Viridis", "Cividis", "Plasma", etc.
colors = px.colors.sample_colorscale(colormap, np.linspace(0, 1, mysoftplankton.Nspheres))

In [57]:
# Calculating eigenvec
eigenvecs, velocities, cos_angles = calculation_eigenvecs(params,delta)

# Plot eigenvectors and direction of descent velocity
n_min = np.argmin(cos_angles)
eigenvec = eigenvecs[n_min]
velocity = velocities[n_min]
angle = 180 * jnp.arccos(cos_angles[n_min]) / jnp.pi

### Plotting figure

In [58]:
fig = go.Figure()

# Plot eigenvectors centered at delta
start = - 2 * eigenvec  # One end of the segment
end = 2 * eigenvec    # The other end of the segment

fig.add_trace(go.Scatter3d(
    x=[start[0], end[0]],
    y=[start[1], end[1]],
    z=[start[2], end[2]],
    mode="lines",
    line=dict(color="green", width=4),
    name="Gravity direction"
))    

start = - 2 * velocity  # One end of the segment
end = 2 * velocity    # The other end of the segment

fig.add_trace(go.Scatter3d(
    x=[start[0], end[0]],
    y=[start[1], end[1]],
    z=[start[2], end[2]],
    mode="lines",
    line=dict(color="red", width=4),
    name="Sinking direction"
))

    
# Plot spheres
for i_sphere, sphere in enumerate(mysoftplankton.spheres):
    x, y, z = create_sphere(
        radius=sphere.radius(dofs, params),
        center=sphere.position(dofs, params),
    )

    # Uniform color per sphere
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        surfacecolor=jnp.ones_like(x) * i_sphere,  # Uniform color
        showscale=False,
        colorscale=[[0, colors[i_sphere]], [1, colors[i_sphere]]],  # Uniform color mapping
        opacity=0.5,  # Transparency level
    ))

    # Dummy scatter point for legend
    fig.add_trace(go.Scatter3d(
        x=[sphere.position(dofs, params)[0]], 
        y=[sphere.position(dofs, params)[1]], 
        z=[sphere.position(dofs, params)[2]],
        mode="markers",
        marker=dict(size=5, color=colors[i_sphere]),
        name=f"Sphere {i_sphere}"  # Legend entry
    ))


# Add geometric center point
fig.add_trace(go.Scatter3d(
    x=[geometric_center[0]], 
    y=[geometric_center[1]], 
    z=[geometric_center[2]],
    mode='markers',
    marker=dict(size=8, color='red', symbol='circle'),
    name="Center of buoyancy"
))

# Add Center of gravity
fig.add_trace(go.Scatter3d(
    x=[delta_point[0]], 
    y=[delta_point[1]], 
    z=[delta_point[2]],
    mode='markers',
    marker=dict(size=8, color='blue', symbol='diamond'),
    name="Center of gravity"
))

fig.update_layout(
    title=f"Sphere assembly with maximum sinking angle of {angle:.2f}°",
    autosize=False,
    width=1100,
    height=900,
    scene=dict(
        xaxis=dict(visible=False, range=[mid_x - max_range, mid_x + max_range]),
        yaxis=dict(visible=False, range=[mid_y - max_range, mid_y + max_range]),
        zaxis=dict(visible=False, range=[mid_z - max_range, mid_z + max_range]),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1),
    ),
    scene_camera=dict(projection=dict(type="orthographic"))  # Isometric projection
)

fig.show()

## Optimizing $\mu_\perp / \mu_\parallel$

### New YAML file

In [67]:
yaml_data = """
param_names:
    - radexp
    - pos
    
defaults:
    radexp1: 0.75
    pos1: 0.3
    pos2: 0.9
    pos3: 0.1
    pos4: 0.9
    # pos5: 0.9

spheres:
  - # Sphere 0
    radius: 1.
    position: [0, 0, 0]
    
  - # Sphere 1
    radius: 0.1 * 10**radexp1
    position: 
      - 10 * (2*pos1 - 1)
      - 0
      - 0
    
  - # Sphere 2
    radius: 0.1 * 10**radexp2
    position: 
      - 5 * (2*pos2 - 1)
      - 0   
      - 0   
    
  - # Sphere 3
    radius: 0.1 * 10**radexp3
    position: 
      - 10 * (2*pos3 - 1)
      - 0   
      - 0   
      
  - # Sphere 4
    radius: 0.1 * 10**radexp4
    position: 
      - 10 * (2*pos4 - 1)
      - 0   
      - 0   
      
  # - # Sphere 5
  #   radius: 0.1 * 10**radexp5
  #   position: 
  #     - 15 * (2*pos5 - 1)
  #     - 0   
  #     - 0         
"""

In [68]:
# Creating SoftPlankton object
mysoftplankton = SoftPlankton(yaml_data, verbose=False)

### Contraints of no complete overlap of spheres

In [51]:
def create_constraint_functions(mysoftplankton):
    """Precompute and JIT-compile g and its Jacobian for a given plankton instance."""
    dofs = jnp.array([])

    def g(params):
        """Return only violated constraints g_ij."""
        positions = jnp.array([s.position(dofs, params) for s in mysoftplankton.spheres])
        radii = jnp.array([s.radius(dofs, params) for s in mysoftplankton.spheres])
        violations = []
        for i in range(len(mysoftplankton.spheres)):
            for j in range(i + 1, mysoftplankton.Nspheres):
                p_i, p_j = positions[i], positions[j]
                r_i, r_j = radii[i], radii[j]
                D_ij = jnp.linalg.norm(p_i - p_j)
                g_ij = (r_i + r_j) - D_ij
                violations.append(g_ij)

        return jnp.array(violations)

    # JIT-compile g and its Jacobian
    g_jit = jax.jit(g)
    jacobian_g_jit = jax.jit(jax.jacfwd(g))

    return g_jit, jacobian_g_jit

In [52]:
# Call this **once** before the optimization starts
g_jit, jacobian_g_jit = create_constraint_functions(mysoftplankton)

@jax.jit
def project_onto_nonoverlap(params):
    """JIT-optimized projection of params to satisfy non-overlapping constraints."""
    step_size = 1.01  # Fixed hyperparameters (JIT requires constants)
    eps = 1e-8
    max_iters = 100

    def cond_fn(state):
        params, i = state
        g_values = g_jit(params)
        return (jnp.max(g_values) > eps) & (i < max_iters)  # Stop early if constraints are satisfied

    def body_fn(state):
        params, i = state
        g_values = g_jit(params)
        J = jacobian_g_jit(params)

        weights = jnp.where(g_values > eps, 1.0, 0.0)  
        J_weighted = J * weights[:, None]  
        g_weighted = g_values * weights

        correction = step_size * (J_weighted.T @ g_weighted) / (1 + jnp.linalg.norm(J_weighted, axis=0))
        new_params = params - correction  

        return new_params, i + 1  

    params, _ = jax.lax.while_loop(cond_fn, body_fn, (params, 0))
    
    return params

### Function `optimize_mu_perp_over_mu_parallel`

In [22]:
def optimize_mu_perp_over_mu_parallel(
    init_parameters,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):

    @jax.jit
    def mu_ratio(params):
        """Computes cos(angle) for given delta and radii."""
        dofs = jnp.array([])
        J = mysoftplankton.compute_composition_of_velocity(dofs, params)
        R = mysoftplankton.compute_resistance_tensor(dofs, params)
        Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
        Mred = jnp.linalg.pinv(J.T @ R @ J)
        M_mean = Mred @ J.T @ Tf[:, :6]

        # Compute submatrices
        mu_tt = M_mean[:3, :3]
        eigvals = jnp.linalg.eigvals(mu_tt)
        eigvals = jnp.real(eigvals) 
        
        # Compute the largest and second largest eigenvalues 
        eigvals, _ = jax.lax.top_k(eigvals, 3)
        lambda_max = eigvals[0]
        lambda_mid = eigvals[1]
        lambda_min = eigvals[2]

        return lambda_min / (lambda_max + 1.E-18)

    def constraints_params(parameters):
        return jnp.clip(parameters, 0, 1)

    parameters = init_parameters
    grad_mu_ratio = jax.jit(jax.grad(mu_ratio))
    velocity = jnp.zeros_like(parameters)  # Initialize velocity for momentum

    for i in range(max_iters):
        grad = grad_mu_ratio(parameters)
        grad_norm = jnp.linalg.norm(grad)

        if i % 1000 == 0:
            print(i, mu_ratio(parameters), parameters, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity - learning_rate * grad
        parameters = parameters + velocity  # Update parameters

        # Apply projection to enforce non-overlapping constraint
        parameters = project_onto_nonoverlap(parameters)

        # Enforce any additional constraints (like bounding params between 0 and 1)
        parameters = constraints_params(parameters)

    return parameters, mu_ratio(parameters)

### Trial run

In [69]:
init_params = mysoftplankton.param_defaults
params, mu_ratio = optimize_mu_perp_over_mu_parallel(
    init_parameters=init_params,
    learning_rate=0.1, 
    max_iters=10000,
)
dofs = jnp.array([])
radii = np.array([mysoftplankton.spheres[i].radius(dofs, params) for i in range(mysoftplankton.Nspheres)])
positions = np.array([mysoftplankton.spheres[i].position(dofs, params) for i in range(mysoftplankton.Nspheres)])
print("\nOptimal radii:", radii)
print("positions:\n", positions)
print("mu_ratio:", 1/mu_ratio)

0 0.93094164 [0.3  0.9  0.1  0.9  0.75 0.   0.   0.  ] 0.84361356


ValueError: too many values to unpack (expected 2)

In [ ]:
1/mu_ratio

Array(1.706, dtype=float32)

In [ ]:
matrices = mysoftplankton.compute_mobility_problem(dofs, params)
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)
print("M")
print(M_mean)

print("M_tt")
print(M_mean[:3,:3])

M
[[ 0.034  0.     0.     0.     0.     0.   ]
 [ 0.     0.05   0.     0.     0.     0.008]
 [ 0.     0.     0.05   0.    -0.008  0.   ]
 [ 0.     0.     0.     0.02   0.     0.   ]
 [ 0.     0.    -0.008  0.     0.003  0.   ]
 [ 0.     0.008  0.     0.     0.     0.003]]
M_tt
[[0.034 0.    0.   ]
 [0.    0.05  0.   ]
 [0.    0.    0.05 ]]


### Plotting figure

In [57]:
# Compute min/max ranges for each axis
all_positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
all_radii = jnp.array([sphere.radius(dofs, params) for sphere in mysoftplankton.spheres])

x_min, x_max = jnp.min(all_positions[:, 0] - all_radii), jnp.max(all_positions[:, 0] + all_radii)
y_min, y_max = jnp.min(all_positions[:, 1] - all_radii), jnp.max(all_positions[:, 1] + all_radii)
z_min, z_max = jnp.min(all_positions[:, 2] - all_radii), jnp.max(all_positions[:, 2] + all_radii)

# Use a consistent range to ensure 1:1:1 aspect ratio
max_range = max(x_max - x_min, y_max - y_min, z_max - z_min) / 2.0
mid_x, mid_y, mid_z = (x_max + x_min) / 2, (y_max + y_min) / 2, (z_max + z_min) / 2

# Choose a colormap
colormap = "Viridis"  # Try "Turbo", "Viridis", "Cividis", "Plasma", etc.
colors = px.colors.sample_colorscale(colormap, np.linspace(0, 1, mysoftplankton.Nspheres))

In [59]:
matrices = mysoftplankton.compute_mobility_problem(dofs, params)
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)
mu_tt = M_mean[:3,:3]

eigvals, eigvecs = jnp.linalg.eigh(mu_tt)
i_sorted = jnp.argsort(eigvals)

eigvals = eigvals[i_sorted]
eigvecs = eigvecs[i_sorted]

In [60]:
fig = go.Figure()
    
# Plot eigvectors 
for eigval, eigvec in zip(eigvals, eigvecs):
    start = - 2 * eigvec  # One end of the segment
    end = 2 * eigvec    # The other end of the segment

    print(eigval, eigvec)

    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line=dict(width=4),
        name=f"Eigenvector with eigenvalue {eigval:.3f}"
    ))    
    
# Plot spheres
for i_sphere, sphere in enumerate(mysoftplankton.spheres):
    x, y, z = create_sphere(
        radius=sphere.radius(dofs, params),
        center=sphere.position(dofs, params),
    )

    # Uniform color per sphere
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        surfacecolor=jnp.ones_like(x) * i_sphere,  # Uniform color
        showscale=False,
        colorscale=[[0, colors[i_sphere]], [1, colors[i_sphere]]],  # Uniform color mapping
        opacity=0.5,  # Transparency level
    ))

    # Dummy scatter point for legend
    fig.add_trace(go.Scatter3d(
        x=[sphere.position(dofs, params)[0]], 
        y=[sphere.position(dofs, params)[1]], 
        z=[sphere.position(dofs, params)[2]],
        mode="markers",
        marker=dict(size=5, color=colors[i_sphere]),
        name=f"Sphere {i_sphere}"  # Legend entry
    ))

fig.update_layout(
    title=f"Sphere assembly",
    autosize=False,
    width=1100,
    height=900,
    scene=dict(
        xaxis=dict(visible=False, range=[mid_x - max_range, mid_x + max_range]),
        yaxis=dict(visible=False, range=[mid_y - max_range, mid_y + max_range]),
        zaxis=dict(visible=False, range=[mid_z - max_range, mid_z + max_range]),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1),
    ),
    scene_camera=dict(projection=dict(type="orthographic"))  # Isometric projection
)

fig.show()

0.034271363 [1. 0. 0.]
0.050080743 [0. 1. 0.]
0.050080743 [0. 0. 1.]


In [ ]:
jnp.array([s.position(dofs, params) for s in mysoftplankton.spheres])


Array([[  0.   ,   0.   ,   0.   ],
       [ -3.   ,   0.   ,   0.   ],
       [  1.796,   0.   ,   0.   ],
       [-10.   ,  -0.688,   0.   ],
       [ 10.   , -10.   ,   0.   ]], dtype=float32)

In [ ]:
is_inside

array([False, False, False, False, False, False, False, False, False, False])

## Optimisation of iso-dense shape

### New YAML file

In [116]:
yaml_data = """
param_names:
    - radexp
    - pos
    
spheres:
  - # Sphere 0
    radius: 1.
    position: [0, 0, 0]
    
  - # Sphere 1
    radius: 1.
    position: 
      - 10 * (2*pos0 - 1)
      - 0
      - 0
    
  - # Sphere 2
    radius: 1.
    position: 
      - 10 * (2*pos1 - 1)
      - 10 * (2*pos2 - 1)
      - 0   
    
  - # Sphere 3
    radius: 1.
    position: 
      - 10 * (2*pos3 - 1)
      - 10 * (2*pos4 - 1)
      - 10 * (2*pos5 - 1)      
      
  - # Sphere 4
    radius: 1.
    position: 
      - 10 * (2*pos6 - 1)
      - 10 * (2*pos7 - 1)
      - 10 * (2*pos8 - 1)    
      
  - # Sphere 5
    radius: 1.
    position: 
      - 10 * (2*pos9 - 1)
      - 10 * (2*pos10 - 1)
      - 10 * (2*pos11 - 1)    
      
  - # Sphere 6
    radius: 1.
    position: 
      - 10 * (2*pos12 - 1)
      - 10 * (2*pos13 - 1)
      - 10 * (2*pos14 - 1) 
      
  - # Sphere 7
    radius: 1.
    position: 
      - 10 * (2*pos15 - 1)
      - 10 * (2*pos16 - 1)
      - 10 * (2*pos17 - 1) 
      
  - # Sphere 8
    radius: 1.
    position: 
      - 10 * (2*pos18 - 1)
      - 10 * (2*pos19 - 1)
      - 10 * (2*pos20 - 1) 
            
  - # Sphere 9
    radius: 1.
    position: 
      - 10 * (2*pos21 - 1)
      - 10 * (2*pos22 - 1)
      - 10 * (2*pos23 - 1)    
"""

In [117]:
# Creating SoftPlankton object
mysoftplankton = SoftPlankton(yaml_data, verbose=False)

### Contraints of no complete overlap of spheres

In [118]:
# Call this **once** before the optimization starts
g_jit, jacobian_g_jit = create_constraint_functions(mysoftplankton)

@jax.jit
def project_onto_nonoverlap(params):
    """JIT-optimized projection of params to satisfy non-overlapping constraints."""
    step_size = 1.01  # Fixed hyperparameters (JIT requires constants)
    eps = 1e-8
    max_iters = 100

    def cond_fn(state):
        params, i = state
        g_values = g_jit(params)
        return (jnp.max(g_values) > eps) & (i < max_iters)  # Stop early if constraints are satisfied

    def body_fn(state):
        params, i = state
        g_values = g_jit(params)
        J = jacobian_g_jit(params)

        weights = jnp.where(g_values > eps, 1.0, 0.0)  
        J_weighted = J * weights[:, None]  
        g_weighted = g_values * weights

        correction = step_size * (J_weighted.T @ g_weighted) / (1 + jnp.linalg.norm(J_weighted, axis=0))
        new_params = params - correction  

        return new_params, i + 1  

    params, _ = jax.lax.while_loop(cond_fn, body_fn, (params, 0))
    
    return params

### Function `optimize_params_isodense` 

In [ ]:
def optimize_params_isodense(
    # init_params,
    num_random_init=100,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):
    """Optimizes both delta and radii using gradient descent with constraints.
    
    - Randomly samples `num_random_params` initial parameter sets.
    - Selects the best initial set by optimizing delta.
    - Performs gradient descent on both delta and parameters.
    """

    # assert len(init_params==mysoftplankton.Nparam), f"`init_params` has not the right lenght {3 + mysoftplankton.Nparam}"
    # params = jnp.array(init_params)
    
    @jax.jit
    def cos_angle(params):
        """Computes cos(angle) for given delta and radii."""
        dofs = jnp.array([])
        J = mysoftplankton.compute_composition_of_velocity(dofs, params)
        R = mysoftplankton.compute_resistance_tensor(dofs, params)
        Tf = mysoftplankton.compute_composition_of_forces(dofs, params)
        Mred = jnp.linalg.inv(J.T @ R @ J)
        M_mean = Mred @ J.T @ Tf[:, :6]

        # Compute submatrices
        mu_tt = M_mean[:3, :3]
        mu_tr = M_mean[:3, 3:]
        mu_rt = M_mean[3:, :3]
        mu_rr = M_mean[3:, 3:]

        # Center of gravity/buoyancy
        volumes = jnp.array([sphere.radius(dofs, params)**3 for sphere in mysoftplankton.spheres])
        positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
        delta = jnp.sum(volumes[:, None] * positions, axis=0) / jnp.sum(volumes)
        delta_cross = jnp.array([
            [0, -delta[2], delta[1]],
            [delta[2], 0, -delta[0]],
            [-delta[1], delta[0], 0],
        ])

        M2 = mu_rt + mu_rr @ delta_cross
        M1 = mu_tt + mu_tr @ delta_cross - delta_cross @ M2

        # Compute eigenvalues
        eigvals = jnp.linalg.eigvals(M2)

        # Thresholds for imaginary and small real parts
        eps_imag = 0.01 *  jnp.max(jnp.abs(jnp.real(eigvals))) # Max allowed imaginary part
        eps_real = 1e-8  # Min allowed real part

        # Mask for nearly real eigenvalues (small imaginary part)
        real_mask = jnp.abs(jnp.imag(eigvals)) < eps_imag

        # Extract real parts
        real_eigvals = jnp.where(real_mask, jnp.real(eigvals), 0.0)

        # Apply a smooth penalty: replace small real eigenvalues smoothly
        penalized_eigvals = jnp.where(
            real_mask, 
            jnp.exp(- jnp.abs(real_eigvals) / eps_real), 
            1.
        )  # Discard non-real ones

        @jax.jit
        def solve_eigenvector(lmbda):
            """Finds an eigenvector using multiple cross products for robustness."""
            A = M2 - lmbda * jnp.eye(3)

            # Compute three different cross products
            v1 = jnp.cross(A[0], A[1])
            v2 = jnp.cross(A[0], A[2])
            v3 = jnp.cross(A[1], A[2])

            # Compute norms
            norm_v1 = jnp.linalg.norm(v1)
            norm_v2 = jnp.linalg.norm(v2)
            norm_v3 = jnp.linalg.norm(v3)

            # Choose the vector with the largest norm
            eigvec = jnp.where(norm_v1 >= norm_v2, v1, v2)
            eigvec = jnp.where(jnp.linalg.norm(eigvec) >= norm_v3, eigvec, v3)

            # Normalize and return
            return eigvec / (jnp.linalg.norm(eigvec) + 1e-16)  # Avoid NaNs

        # Compute eigenvectors for penalized eigenvalues
        real_eigvecs = jax.vmap(solve_eigenvector)(real_eigvals)
        M1_eigvecs = jnp.einsum("ij,kj->ik", M1, real_eigvecs)

        cos_angles = jnp.einsum("ij,ji->i", real_eigvecs, M1_eigvecs) / jnp.linalg.norm(
            M1_eigvecs, axis=0
        )
        cos_angles = jnp.real(cos_angles) + penalized_eigvals

        # Find minimum cos(angle) after penalization
        min_cos_angle = jnp.min(jnp.where(real_mask, cos_angles, 2.0))  # Set invalid ones to 2.0 (out of range)

        return min_cos_angle

    def constraints_params(params):
        """Clips radii and enforces delta constraints within the convex hull of spheres."""

        # Clip params within [0, 1]
        params = jnp.clip(params, 0, 1)

        # Apply projection to enforce non-overlapping constraint
        params = project_onto_nonoverlap(params)

        return params

    # Generate multiple random initializations
    random_inits = jax.random.uniform(jax.random.PRNGKey(0), shape=(num_random_init, mysoftplankton.Nparam), minval=0, maxval=1)

    # Project onto the feasible space
    projected_params = jax.vmap(project_onto_nonoverlap)(random_inits)

    # Evaluate the initial cos_angle for each set
    cos_angles = jax.vmap(cos_angle)(projected_params)

    # Select the best starting parameters (smallest cos_angle)
    best_idx = jnp.argmin(cos_angles)
    params = projected_params[best_idx]

    print(f"Selected best initialization with cos_angle={cos_angles[best_idx]:.6f}")

    # Initialize best parameters tracking
    best_parameters = params
    best_cos_angle = cos_angle(params)
    grad_threshold = 0.001  # Threshold for smoothing

    grad_cos_angle = jax.jit(jax.grad(cos_angle))
    velocity = jnp.zeros_like(params)  # Initialize velocity for momentum

    for i in range(max_iters):
        grad = grad_cos_angle(params)
        grad_norm = jnp.linalg.norm(grad)

        # Smoothly scale gradient if too large
        scaling_factor = jnp.minimum(1.0, grad_threshold / (grad_norm + 1e-8))  # Smooth rescaling
        grad = grad * scaling_factor  

        # Compute new parameters
        velocity = momentum * velocity - learning_rate * grad
        params = params + velocity  # Update parameters

        # Enforce constraints
        params = constraints_params(params)
        
        # Compute current cos_angle
        current_cos_angle = cos_angle(params)

        # Track the best parameters based on minimum cos_angle
        if current_cos_angle < best_cos_angle:
            best_cos_angle = current_cos_angle
            best_parameters = params

        # Screen display of iterative process
        if i % 1000 == 0:
            param_str = (
                f"{params[:2]} ... {params[-2:]}" if len(params) > 4 else str(params)
            )
            print(f"Iter: {i:5d} | Best cos_angle: {best_cos_angle:.6f} | "
                f"Current cos_angle: {current_cos_angle:.6f} | "
                f"Params: {param_str} | Grad norm: {grad_norm:.4f}")

    return best_parameters, best_cos_angle

### Trial run

In [120]:
params, mu_ratio = optimize_params_isodense(
    # init_params=mysoftplankton.param_defaults,
    learning_rate=0.1, 
    max_iters=100000,
)
dofs = jnp.array([])
radii = np.array([mysoftplankton.spheres[i].radius(dofs, params) for i in range(mysoftplankton.Nspheres)])
positions = np.array([mysoftplankton.spheres[i].position(dofs, params) for i in range(mysoftplankton.Nspheres)])
print("\nOptimal radii:", radii)
print("positions:\n", positions)
print("mu_ratio:", mu_ratio)

Selected best initialization with cos_angle=0.998139
Iter:     0 | Best cos_angle: 0.998137 | Current cos_angle: 0.998137 | Params: [0.601 0.639] ... [0.314 0.255] | Grad norm: 0.012236
Iter:  1000 | Best cos_angle: 0.986120 | Current cos_angle: 0.986120 | Params: [0.622 0.523] ... [0.366 0.273] | Grad norm: 0.020135
Iter:  2000 | Best cos_angle: 0.985889 | Current cos_angle: 0.985889 | Params: [0.614 0.506] ... [0.393 0.246] | Grad norm: 0.068256
Iter:  3000 | Best cos_angle: 0.985816 | Current cos_angle: 0.985816 | Params: [0.61  0.496] ... [0.404 0.235] | Grad norm: 0.092354
Iter:  4000 | Best cos_angle: 0.985768 | Current cos_angle: 0.985768 | Params: [0.608 0.488] ... [0.412 0.227] | Grad norm: 0.109376
Iter:  5000 | Best cos_angle: 0.985734 | Current cos_angle: 0.985734 | Params: [0.607 0.482] ... [0.418 0.221] | Grad norm: 0.123546
Iter:  6000 | Best cos_angle: 0.985709 | Current cos_angle: 0.985709 | Params: [0.605 0.477] ... [0.423 0.216] | Grad norm: 0.136525
Iter:  7000 | Be

In [121]:
dofs = jnp.array([])
volumes = jnp.array([sphere.radius(dofs, params)**3 for sphere in mysoftplankton.spheres])
positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
delta = jnp.sum(volumes[:, None] * positions, axis=0) / jnp.sum(volumes)

print(calculation_angle(params, delta))

Matrix M1
 [[ 1.469e-02  2.224e-03 -1.257e-05]
 [ 2.224e-03  1.337e-02 -1.013e-05]
 [-1.257e-05 -1.013e-05  1.160e-02]]
Matrix M2
 [[ 4.058e-06 -5.519e-06  1.036e-07]
 [ 4.538e-06 -2.988e-06  1.656e-06]
 [ 9.424e-07 -9.565e-07 -3.397e-08]]
Eigenvalues of M1: [0.012 0.012 0.016]
Eigenvalues of M2: [ 6.134e-07+3.771e-06j  6.134e-07-3.771e-06j -1.906e-07+0.000e+00j]
Eigenvalues of M2 (real only): [-1.906e-07+0.j]
Cos angles: [0.985]
Norm of eigenvectors [1.]
Eigenvectors [[-0.516+0.j]
 [-0.383+0.j]
 [ 0.767+0.j]]
Angles by inverting cos, sin, tan:
[9.779] °
[9.779] °
[9.779] °
Matrix M1
 [[ 1.469e-02  2.224e-03 -1.257e-05]
 [ 2.224e-03  1.337e-02 -1.013e-05]
 [-1.257e-05 -1.013e-05  1.160e-02]]
Matrix M2
 [[ 4.058e-06 -5.519e-06  1.036e-07]
 [ 4.538e-06 -2.988e-06  1.656e-06]
 [ 9.424e-07 -9.565e-07 -3.397e-08]]
Eigenvalues of M2: [ 6.134e-07+3.771e-06j  6.134e-07-3.771e-06j -1.906e-07+0.000e+00j]
Eigenvalues of M2 (real only): [ 0.000e+00  0.000e+00 -1.906e-07]
Eigenvectors [[-0.524 -0.3

## Visualization of the sphere assembly

In [122]:
# Compute geometric center (volume-weighted) / aka center of buoyancy
volumes = jnp.array([4/3 * jnp.pi * sphere.radius(dofs, params)**3 for sphere in mysoftplankton.spheres])
positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
geometric_center = jnp.sum(volumes[:, None] * positions, axis=0) / jnp.sum(volumes)

# Compute min/max ranges for each axis
all_positions = jnp.array([sphere.position(dofs, params) for sphere in mysoftplankton.spheres])
all_radii = jnp.array([sphere.radius(dofs, params) for sphere in mysoftplankton.spheres])

x_min, x_max = jnp.min(all_positions[:, 0] - all_radii), jnp.max(all_positions[:, 0] + all_radii)
y_min, y_max = jnp.min(all_positions[:, 1] - all_radii), jnp.max(all_positions[:, 1] + all_radii)
z_min, z_max = jnp.min(all_positions[:, 2] - all_radii), jnp.max(all_positions[:, 2] + all_radii)

# Use a consistent range to ensure 1:1:1 aspect ratio
max_range = max(x_max - x_min, y_max - y_min, z_max - z_min) / 2.0
mid_x, mid_y, mid_z = (x_max + x_min) / 2, (y_max + y_min) / 2, (z_max + z_min) / 2

# Choose a colormap
colormap = "Viridis"  # Try "Turbo", "Viridis", "Cividis", "Plasma", etc.
colors = px.colors.sample_colorscale(colormap, np.linspace(0, 1, mysoftplankton.Nspheres))

In [123]:
# Calculating eigenvec
eigenvecs, velocities, cos_angles = calculation_eigenvecs(params,delta)

In [124]:
fig = go.Figure()

for eigenvec, velocity, cos_angle in zip(eigenvecs, velocities, cos_angles):
    # Plot eigenvectors and direction of descent velocity
    angle = 180 * jnp.arccos(cos_angle) / jnp.pi

    # Plot eigenvectors centered at delta
    start = - 2 * eigenvec  # One end of the segment
    end = 2 * eigenvec    # The other end of the segment

    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line=dict(color="green", width=4),
        name="Eigen vector"
    ))    

    start = - 2 * velocity  # One end of the segment
    end = 2 * velocity    # The other end of the segment

    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line=dict(color="red", width=4),
        name=f"Sinking direction with angle {angle:.2f}"
    ))

    
# Plot spheres
for i_sphere, sphere in enumerate(mysoftplankton.spheres):
    x, y, z = create_sphere(
        radius=sphere.radius(dofs, params),
        center=sphere.position(dofs, params),
    )

    # Uniform color per sphere
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        surfacecolor=jnp.ones_like(x) * i_sphere,  # Uniform color
        showscale=False,
        colorscale=[[0, colors[i_sphere]], [1, colors[i_sphere]]],  # Uniform color mapping
        opacity=0.5,  # Transparency level
    ))

    # Dummy scatter point for legend
    fig.add_trace(go.Scatter3d(
        x=[sphere.position(dofs, params)[0]], 
        y=[sphere.position(dofs, params)[1]], 
        z=[sphere.position(dofs, params)[2]],
        mode="markers",
        marker=dict(size=5, color=colors[i_sphere]),
        name=f"Sphere {i_sphere}"  # Legend entry
    ))


# Add geometric center point
fig.add_trace(go.Scatter3d(
    x=[geometric_center[0]], 
    y=[geometric_center[1]], 
    z=[geometric_center[2]],
    mode='markers',
    marker=dict(size=8, color='red', symbol='circle'),
    name="Center of buoyancy"
))

fig.update_layout(
    title=f"Sphere assembly with maximum sinking angle of {angle:.2f}°",
    autosize=False,
    width=1100,
    height=900,
    scene=dict(
        xaxis=dict(visible=False, range=[mid_x - max_range, mid_x + max_range]),
        yaxis=dict(visible=False, range=[mid_y - max_range, mid_y + max_range]),
        zaxis=dict(visible=False, range=[mid_z - max_range, mid_z + max_range]),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1),
    ),
    scene_camera=dict(projection=dict(type="orthographic"))  # Isometric projection
)

fig.show()

## Notes

In [ ]:
from itertools import combinations

def project_onto_convex_hull(delta, positions):
    """Projects delta onto an approximate convex hull using triangle faces."""
    positions = jnp.array(positions)
    assembly_center = jnp.mean(positions, axis=0)  # Center of mass

    to_delta = delta - assembly_center

    best_projection = delta
    min_distance = jnp.inf
    max_distance = -jnp.inf

    for i, j, k in combinations(range(len(positions)), 3):
        p1, p2, p3 = positions[i], positions[j], positions[k]
        triangle_center = (p1 + p2 + p3) / 3

        # Compute normal of the triangle
        normal = jnp.cross(p2 - p1, p3 - p1)
        normal *= np.dot(normal, triangle_center - assembly_center)
        normal = normal / jnp.linalg.norm(normal)

        # Compute the intersection of the line from the assembly center through delta with the plane
        distance = jnp.dot(triangle_center - assembly_center, normal) / jnp.dot(to_delta, normal)
        intersection = assembly_center + distance * to_delta
        
        # Check if the intersection is inside the triangle
        def is_inside_triangle(pt, p1, p2, p3):
            # Barycentric coordinates check
            v0 = p3 - p1
            v1 = p2 - p1
            v2 = pt - p1

            d00 = jnp.dot(v0, v0)
            d01 = jnp.dot(v0, v1)
            d11 = jnp.dot(v1, v1)
            d20 = jnp.dot(v2, v0)
            d21 = jnp.dot(v2, v1)

            denom = d00 * d11 - d01 * d01
            v = (d11 * d20 - d01 * d21) / denom
            w = (d00 * d21 - d01 * d20) / denom
            u = 1.0 - v - w

            # Check if the point lies inside the triangle (barycentric coordinates >= 0)
            return u >= 0 and v >= 0 and w >= 0

        # If the intersection point lies inside the triangle
        if is_inside_triangle(intersection, p1, p2, p3):

            if distance < min_distance:
                min_distance = distance
                
            if distance > max_distance:
                max_distance = distance

    # If delta in between the extrme the original delta
    if max_distance > 1 and min_distance < 1:
        return delta
    elif max_distance > 1:
        return assembly_center + min_distance * to_delta
    else:
        return assembly_center + max_distance * to_delta

In [ ]:
# Generate random sphere positions
num_spheres = 10
positions = np.random.uniform(-1, 1, (num_spheres, 3))

# Generate random deltas
num_deltas = 10
deltas = np.random.uniform(-1., 1., (num_deltas, 3))

# Compute projections and classify points
projected_deltas = np.array([project_onto_convex_hull(d, positions) for d in deltas])
is_inside = np.isclose(np.linalg.norm(deltas - projected_deltas, axis=1), 0, atol=1e-3)


0 1 5 inf -0.161541 -inf
0 1 7 -0.161541 -0.15977347 -0.161541
0 1 8 -0.161541 -0.17692785 -0.15977347
0 1 9 -0.17692785 -0.13302168 -0.15977347
0 3 5 -0.17692785 -0.32628536 -0.13302168
0 3 9 -0.32628536 -0.4163734 -0.13302168
0 4 5 -0.4163734 -0.12296461 -0.13302168
0 5 6 -0.4163734 -0.0515656 -0.12296461
0 6 7 -0.4163734 0.055991266 -0.0515656
0 6 8 -0.4163734 -0.048073195 0.055991266
0 6 9 -0.4163734 0.21004215 0.055991266
1 2 5 -0.4163734 0.4118896 0.21004215
1 2 7 -0.4163734 0.2953143 0.4118896
1 2 8 -0.4163734 0.27943847 0.4118896
1 2 9 -0.4163734 0.38631263 0.4118896
1 3 7 -0.4163734 -0.35249975 0.4118896
1 3 8 -0.4163734 -0.3832392 0.4118896
1 4 7 -0.4163734 -0.121449314 0.4118896
1 4 8 -0.4163734 -0.16775657 0.4118896
1 4 9 -0.4163734 0.0948002 0.4118896
2 3 5 -0.4163734 0.24349847 0.4118896
2 3 9 -0.4163734 -0.22850265 0.4118896
2 4 5 -0.4163734 0.39899814 0.4118896
2 5 6 -0.4163734 0.5350221 0.4118896
2 6 7 -0.4163734 0.20271847 0.5350221
2 6 8 -0.4163734 0.16892001 0.53502

In [ ]:

# Plot spheres, deltas, and projected points
fig = go.Figure()

# Plot sphere positions
fig.add_trace(go.Scatter3d(x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
                           mode='markers', marker=dict(size=6, color='black'), name='Spheres'))

# Compute and plot the convex hull
hull = ConvexHull(positions)

for simplex in hull.simplices:
    # Get the vertices of the current face (triangle)
    v0, v1, v2 = positions[simplex]
    
    # Create a face (triangle) for the convex hull
    fig.add_trace(go.Mesh3d(x=[v0[0], v1[0], v2[0]], y=[v0[1], v1[1], v2[1]], z=[v0[2], v1[2], v2[2]],
                           color='rgba(255, 0, 0, 0.3)', opacity=0.5, name="Convex Hull Face"))


# Plot deltas (red if outside, green if inside)
for i in range(num_deltas):
    color = 'green' if is_inside[i] else 'red'
    fig.add_trace(go.Scatter3d(x=[deltas[i, 0]], y=[deltas[i, 1]], z=[deltas[i, 2]],
                               mode='markers', marker=dict(size=5, color=color),
                               name=f'Delta {i}'))
    
    if not is_inside[i]:  # Only draw projections for outside points
        fig.add_trace(go.Scatter3d(x=[deltas[i, 0], projected_deltas[i, 0]],
                                   y=[deltas[i, 1], projected_deltas[i, 1]],
                                   z=[deltas[i, 2], projected_deltas[i, 2]],
                                   mode='lines', line=dict(color='blue', width=2),
                                   name=f'Projection {i}'))
        fig.add_trace(go.Scatter3d(x=[projected_deltas[i, 0]], y=[projected_deltas[i, 1]],
                                   z=[projected_deltas[i, 2]], mode='markers',
                                   marker=dict(size=5, color='blue'),
                                   name=f'Projected Delta {i}'))

fig.update_layout(title='Convex Hull Projection Test',
                  width=1100,
                  height=900,
                  scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.show()

# Test 2 spheres

In [ ]:
yaml_data = """
param_names:
    - pos
    
defaults:
    pos: 4

spheres:
  - # Sphere 0
    radius: 1
    position: [0, 0, 0]
    
  - # Sphere 1
    radius: 1
    position: [pos, 0, 0]
"""

In [ ]:
# Creating SoftPlankton object
mysoftplankton = SoftPlankton(yaml_data, verbose=True)

  Found variables: pos
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 1
      Radius: 1
      Position: ['pos', '0', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']


In [ ]:
# Rigid mobility matrix
matrices = mysoftplankton.compute_mobility_problem()
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)

# Hydrodynamic mobility center
a = M_mean[:3,:3]
b = M_mean[3:, :3]
c = M_mean[3:, 3:]

levicivita = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# From Kim & Karila 5.3.2
trcmcinv = np.linalg.inv(np.trace(c) * np.eye(3) - c)
x_cm = 0.5 * np.einsum("ij,jkl,kl->i", trcmcinv, levicivita, b - b.transpose())

# Compute delta_cross (cross-product matrix)
x_cm_cross = jnp.array([
    [0, -x_cm[2], x_cm[1]],
    [x_cm[2], 0, -x_cm[0]],
    [-x_cm[1], x_cm[0], 0]
])
M2 = b + c @ x_cm_cross
M1 = a + b.T @ x_cm_cross - x_cm_cross @ M2

print(6 * jnp.pi * M1)

[[0.68  0.    0.   ]
 [0.    0.596 0.   ]
 [0.    0.    0.596]]


In [ ]:
matrices = mysoftplankton.compute_mobility_problem()
M = matrices.M
M_mean = jnp.mean(
    M.reshape(6, 6 * mysoftplankton.Nspheres)
    .T.reshape(mysoftplankton.Nspheres, 6, 6),
    axis=0,
)
#print(6 * jnp.pi * M)
print(6 * jnp.pi * M_mean)

[[ 6.797e-01  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00]
 [ 0.000e+00  5.962e-01  0.000e+00  0.000e+00  0.000e+00 -1.756e-08]
 [ 0.000e+00  0.000e+00  5.962e-01  0.000e+00  3.511e-08  0.000e+00]
 [ 0.000e+00  0.000e+00  0.000e+00  3.809e-01  0.000e+00  0.000e+00]
 [ 0.000e+00  0.000e+00  1.659e-08  0.000e+00  8.301e-02  0.000e+00]
 [ 0.000e+00 -6.572e-09  0.000e+00  0.000e+00  0.000e+00  8.301e-02]]


In [ ]:
M_G = mysoftplankton.compute_mobility_tensor()
print(6 * jnp.pi * M_G)

[[ 1.     0.     0.     0.     0.     0.     0.359  0.     0.     0.     0.     0.   ]
 [ 0.     1.     0.     0.     0.     0.     0.     0.195  0.     0.     0.    -0.047]
 [ 0.     0.     1.     0.     0.     0.     0.     0.     0.195  0.     0.047  0.   ]
 [ 0.     0.     0.     0.75   0.     0.     0.     0.    -0.     0.012  0.     0.   ]
 [ 0.     0.     0.     0.     0.75   0.     0.     0.    -0.047  0.    -0.006  0.   ]
 [ 0.     0.     0.     0.     0.     0.75   0.     0.047 -0.     0.     0.    -0.006]
 [ 0.359  0.     0.     0.     0.     0.     1.     0.     0.     0.     0.     0.   ]
 [ 0.     0.195  0.     0.     0.     0.047  0.     1.     0.     0.     0.     0.   ]
 [ 0.     0.     0.195 -0.    -0.047 -0.     0.     0.     1.     0.     0.     0.   ]
 [ 0.     0.     0.     0.012  0.     0.     0.     0.     0.     0.75   0.     0.   ]
 [ 0.     0.     0.047  0.    -0.006  0.     0.     0.     0.     0.     0.75   0.   ]
 [ 0.    -0.047  0.     0.     0.    -0.006

In [ ]:
dofs=mysoftplankton.dof_defaults
params=mysoftplankton.param_defaults
r = 2 * params[0]
C1 = 0.75 / r + 0.5 / r**3
C2 = 0.75 / r - 1.5 / r**3
print(C1+C2, C1)


0.359375 0.1953125


In [ ]:
print((1 + C1 + C2)/2 , (1 + C1) / 2)

0.6796875 0.59765625
